In [8]:
import numpy as np

# Geometric parameters
B1 = np.sqrt(2)
L1 = 1.0
L2 = 1.0

# Joint angles in radians (e.g., theta = 90 deg, phi = 90 deg, varphi = 45 deg)
theta = np.deg2rad(90.0)   # Base azimuth rotation
phi = np.deg2rad(90.0)     # Shoulder elevation
varphi = np.deg2rad(45.0) # Elbow relative pitch


def rot_z(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([
        [c, -s, 0.0, 0.0],
        [s,  c, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0]
    ])


def rot_y(angle):
    c, s = np.cos(angle), np.sin(angle)
    return np.array([
        [ c, 0.0, s, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [-s, 0.0, c, 0.0],
        [0.0, 0.0, 0.0, 1.0]
    ])


def trans_x(d):
    return np.array([
        [1.0, 0.0, 0.0,   d],
        [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0]
    ])


def trans_z(d):
    return np.array([
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0,   d],
        [0.0, 0.0, 0.0, 1.0]
    ])


# -------------------------------------------------------------------------
# Sequential Elementary Kinematic Chain:
# 1. Elevate to shoulder joint: Trans_z(B1)
# 2. Base azimuth yaw:          Rot_z(theta)
# 3. Shoulder pitch elevation:  Rot_y(-phi)   [negative for pitching up from +X toward +Z]
# 4. Translate along upper arm: Trans_x(L1)
# 5. Elbow pitch rotation:      Rot_y(-varphi)
# 6. Translate along forearm:   Trans_x(L2)
# -------------------------------------------------------------------------

M1_trans_base     = trans_z(B1)
M2_rot_theta      = rot_z(theta)
M3_rot_phi        = rot_y(-phi)
M4_trans_arm1     = trans_x(L1)
M5_rot_varphi     = rot_y(-varphi)
M6_trans_arm2     = trans_x(L2)

# Direct sequential multiplication
T_total = (
    M1_trans_base
    @ M2_rot_theta
    @ M3_rot_phi
    @ M4_trans_arm1
    @ M5_rot_varphi
    @ M6_trans_arm2
)

# Extract end-effector coordinate
p_origin = np.array([0.0, 0.0, 0.0, 1.0])
p_ee_sequential = (T_total @ p_origin)[:3]

# Direct geometric reference
r_planar = L1 * np.cos(phi) + L2 * np.cos(phi + varphi)
p_ee_geom = np.array([
    r_planar * np.cos(theta),
    r_planar * np.sin(theta),
    B1 + L1 * np.sin(phi) + L2 * np.sin(phi + varphi)
])

print("T_total Matrix:\n", np.round(T_total, 4))
print("\nEnd-Effector (DH parameter):", np.round(p_ee_sequential, 4))
print("End-Effector (Analytical):       ", np.round(p_ee_geom, 4))
print("Match:", np.allclose(p_ee_sequential, p_ee_geom))

T_total Matrix:
 [[-0.     -1.     -0.     -0.    ]
 [-0.7071  0.     -0.7071 -0.7071]
 [ 0.7071  0.     -0.7071  3.1213]
 [ 0.      0.      0.      1.    ]]

End-Effector (DH parameter): [-0.     -0.7071  3.1213]
End-Effector (Analytical):        [-0.     -0.7071  3.1213]
Match: True
